In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Einführung in'n Qiskit AI-gestützten Transpiler-Service

*Geschätzte QPU-Nutzung: Keine (HINWEIS: In dem Tutorial werden keine Jobs ausgeführt, weil's sich auf Transpilation konzentriert)*

## Hintergrund
Da **Qiskit AI-gestützte Transpiler-Service (QTS)** führt auf maschinellem Lernen basierende Optimierungen in Routing- und Synthese-Passes ein. Diese KI-Modi wurden entwickelt, um de Einschränkungen von der traditionellen Transpilation anzugehen, insbesondere bei groß angelegten Schaltkreisen und komplexen Hardware-Topologien.

Ab **Juli 2025** wurde da **Transpiler Service** auf de neue IBM Quantum&reg; Plattform migriert und ist net mehr verfügbar. Für die neuesten Updates zum Status vom Transpiler Service schau dir bitte de [Transpiler-Service-Dokumentation](/guides/qiskit-transpiler-service) an. Du kannst den KI-Transpiler nach wie vor lokal verwenden, ähnlich wie de Standard-Qiskit-Transpilation. Ersetze einfach `generate_preset_pass_manager()` durch `generate_ai_pass_manager()`. Diese Funktion baut a'n Pass Manager, der de KI-gestützten Routing- und Synthese-Passes direkt in deinen lokalen Transpilations-Workflow integriert.

### Wichtigste Funktionen von den KI-Passes
- Routing-Passes: KI-gestütztes Routing kann Qubit-Pfade dynamisch anpassen, je nach'm spezifischen Schaltkreis und Backend, was de Notwendigkeit für übermäßig viele SWAP-Gates reduziert.
    - `AIRouting`: Layout-Auswahl und Schaltkreis-Routing

- Synthese-Passes: KI-Techniken optimieren de Zerlegung von Multi-Qubit-Gates und minimieren dabei de Anzahl von Zwei-Qubit-Gates, die typischerweise fehleranfälliger san.
    - `AICliffordSynthesis`: Clifford-Gate-Synthese
    - `AILinearFunctionSynthesis`: Synthese von linearen Funktionsschaltkreisen
    - `AIPermutationSynthesis`: Permutationsschaltkreis-Synthese
    - `AIPauliNetworkSynthesis`: Pauli-Netzwerk-Schaltkreis-Synthese (nur im Qiskit Transpiler Service verfügbar, net in der lokalen Umgebung)

- Vergleich mit traditioneller Transpilation: Da Standard-Qiskit-Transpiler is a robustes Werkzeug, das a breites Spektrum an Quantenschaltkreisen effektiv verarbeiten kann. Wenn aber Schaltkreise größer werden oder Hardware-Konfigurationen komplexer, können KI-Passes zusätzliche Optimierungsgewinne liefern. Durch den Einsatz von gelernten Modellen für Routing und Synthese verfeinert QTS Schaltkreis-Layouts weiter und reduziert den Overhead bei anspruchsvollen oder groß angelegten Quantenaufgaben.

In dem Tutorial werden die KI-Modi mit sowohl Routing- als auch Synthese-Passes evaluiert, wobei de Ergebnisse mit traditioneller Transpilation verglichen werden, um zu zeigen, wo KI Leistungsgewinne bringt.

Für mehr Details zu den verfügbaren KI-Passes schau dir de [KI-Passes-Dokumentation](/guides/ai-transpiler-passes) an.

### Warum KI für die Transpilation von Quantenschaltkreisen verwenden?
Wenn Quantenschaltkreise in Größe und Komplexität wachsen, haben traditionelle Transpilationsmethoden Schwierigkeiten, Layouts zu optimieren und Gate-Anzahlen effizient zu reduzieren. Größere Schaltkreise, insbesondere solche mit Hunderten von Qubits, stellen beim Routing und bei der Synthese erhebliche Herausforderungen dar, wegen Geräte-Beschränkungen, begrenzter Konnektivität und Qubit-Fehlerraten.

Genau da bietet KI-gestützte Transpilation a potenzielle Lösung. Durch den Einsatz von Techniken des maschinellen Lernens kann da KI-gestützte Transpiler in Qiskit klügere Entscheidungen beim Qubit-Routing und bei der Gate-Synthese treffen, was zu besserer Optimierung von groß angelegten Quantenschaltkreisen führt.

### Kurze Benchmarking-Ergebnisse
![Graph der die KI-Transpiler-Performance gegen Qiskit zeigt](../docs/images/tutorials/ai-transpiler-introduction/ai-transpiler-benchmarks.avif)

In Benchmarking-Tests hat da KI-Transpiler konsistent flachere, qualitativ hochwertigere Schaltkreise produziert als da Standard-Qiskit-Transpiler. Für diese Tests wurde Qiskits Standard-Pass-Manager-Strategie verwendet, konfiguriert mit [`generate_preset_passmanager`]. Obwohl diese Standard-Strategie oft effektiv is, kann sie bei größeren oder komplexeren Schaltkreisen an ihre Grenzen stoßen. Im Vergleich dazu haben KI-gestützte Passes a durchschnittliche Reduzierung von 24 % bei Zwei-Qubit-Gate-Anzahlen und a Reduzierung von 36 % bei der Schaltkreistiefe für große Schaltkreise (100+ Qubits) erreicht, wenn zur Heavy-Hex-Topologie von IBM Quantum Hardware transpiliert wurde. Für mehr Infos zu diesen Benchmarks schau dir diesen [Blog-Beitrag](https://www.ibm.com/quantum/blog/qiskit-performance) an.

In dem Tutorial werden de wichtigsten Vorteile von KI-Passes und ihr Vergleich mit traditionellen Methoden erkundet.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.random import random_circuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
import matplotlib.pyplot as plt
from statistics import mean, stdev
import time
import logging

seed = 42


def transpile_with_metrics(pass_manager, circuit):
    """Transpile a circuit and return the result along with key metrics."""
    start = time.time()
    qc_out = pass_manager.run(circuit)
    elapsed = time.time() - start

    depth_2q = qc_out.depth(lambda x: x.operation.num_qubits == 2)
    gate_count = qc_out.size()

    return qc_out, {
        "depth_2q": depth_2q,
        "gate_count": gate_count,
        "time_s": round(elapsed, 3),
    }


def remap_to_contiguous(tqc):
    """Remap a transpiled circuit to use contiguous qubit indices.

    Transpiled circuits target specific physical qubits (e.g., qubit 45, 67)
    on a large backend. This remaps them to 0, 1, 2, ... so Aer only
    simulates the active qubits."""
    active = sorted(
        {tqc.find_bit(q).index for inst in tqc.data for q in inst.qubits}
    )
    qubit_map = {old: new for new, old in enumerate(active)}
    new_qc = QuantumCircuit(len(active))
    for inst in tqc.data:
        old_indices = [tqc.find_bit(q).index for q in inst.qubits]
        new_qc.append(inst.operation, [qubit_map[i] for i in old_indices])
    return new_qc


def build_mirror_circuit(tqc, simulate=True):
    """Build a mirror circuit: U followed by U-dagger, with measurements.

    The expected output is always |0...0>, so measuring the survival
    probability reveals how much noise each transpilation strategy adds.

    Args:
        tqc: A transpiled circuit.
        simulate: If True (default), remap to contiguous qubits so Aer
            only simulates the active qubits. If False, keep the full
            physical layout for hardware execution."""
    if simulate:
        tqc = remap_to_contiguous(tqc)
    mirror = tqc.compose(tqc.inverse())
    mirror.measure_all()
    return mirror


def print_summary(results):
    """Print a summary of each metric as mean +/- stdev across all circuits,
    along with the mean percentage improvement of AI over Default."""
    metrics = [
        ("Depth 2Q", "Depth 2Q (Default)", "Depth 2Q (AI)"),
        ("Gate Count", "Gate Count (Default)", "Gate Count (AI)"),
        ("Time (s)", "Time (Default)", "Time (AI)"),
    ]
    header = (
        f"{'Metric':<12}{'Default (mean +/- std)':>24}"
        f"{'AI (mean +/- std)':>22}{'AI % improvement':>22}"
    )
    print(header)
    print("-" * len(header))
    for label, col_def, col_ai in metrics:
        defaults = [r[col_def] for r in results]
        ais = [r[col_ai] for r in results]
        pct = [(d - a) / d * 100 for d, a in zip(defaults, ais)]
        default_str = f"{mean(defaults):.1f} +/- {stdev(defaults):.1f}"
        ai_str = f"{mean(ais):.1f} +/- {stdev(ais):.1f}"
        pct_str = f"{mean(pct):+.1f}% +/- {stdev(pct):.1f}%"
        print(f"{label:<12}{default_str:>24}{ai_str:>22}{pct_str:>22}")


def plot_metrics_and_pct(results, title_prefix):
    """Plot metric comparisons and percentage improvement of AI over Default."""
    qubits = [r["Qubits"] for r in results]
    metrics = [
        ("Depth 2Q (Default)", "Depth 2Q (AI)", "Two-Qubit Depth"),
        ("Gate Count (Default)", "Gate Count (AI)", "Gate Count"),
        ("Time (Default)", "Time (AI)", "Transpilation Time"),
    ]

    # Row 1: raw metric comparison
    fig, axs = plt.subplots(1, 3, figsize=(21, 5))
    fig.suptitle(
        f"{title_prefix}: Metric Comparison",
        fontsize=15,
        fontweight="bold",
        y=1.02,
    )
    for ax, (col_def, col_ai, label) in zip(axs, metrics):
        ax.plot(qubits, [r[col_def] for r in results], "o-", label="Default")
        ax.plot(qubits, [r[col_ai] for r in results], "s-", label="AI")
        ax.set_title(label)
        ax.set_xlabel("Number of Qubits")
        ax.set_ylabel(label)
        ax.legend()
    plt.tight_layout()
    plt.show()

    # Row 2: percentage improvement
    fig, axs = plt.subplots(1, 3, figsize=(21, 5))
    fig.suptitle(
        f"{title_prefix}: % Improvement of AI over Default",
        fontsize=15,
        fontweight="bold",
        y=1.02,
    )
    for ax, (col_def, col_ai, label) in zip(axs, metrics):
        pct = [(r[col_def] - r[col_ai]) / r[col_def] * 100 for r in results]
        ax.axhline(
            0, color="#1f77b4", linewidth=2, label="Default (baseline)"
        )
        ax.plot(qubits, pct, "s-", color="#ff7f0e", label="AI")
        ax.fill_between(qubits, 0, pct, alpha=0.15, color="#ff7f0e")
        ax.set_title(label)
        ax.set_xlabel("Number of Qubits")
        ax.set_ylabel("% Improvement")
        ax.legend()
    plt.tight_layout()
    plt.show()


# Suppress verbose AI-powered transpiler logs
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)

## Voraussetzungen

Bevor du mit dem Tutorial anfängst, stell sicher, dass du folgendes installiert hast:

* Qiskit SDK v1.0 oder neuer, mit Unterstützung für [Visualisierung](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime (`pip install qiskit-ibm-runtime`) v0.22 oder neuer
* Qiskit IBM&reg; Transpiler mit KI-Lokalmodus (`pip install 'qiskit-ibm-transpiler[ai-local-mode]'`)

## Setup

In [2]:
num_circuits_sim = 20
depth_sim = 4
qubit_range_sim = list(range(6, 26))

circuits_sim = [
    # We have only two qubit gates, as those test how well the
    # transpiler can optimize the circuit.
    random_circuit(
        num_qubits=n,
        depth=depth_sim,
        max_operands=2,
        num_operand_distribution={2: 1},
        seed=seed + i,
    )
    for i, n in enumerate(qubit_range_sim)
]

print(
    f"Created {len(circuits_sim)} circuits with qubit counts "
    f"from {qubit_range_sim[0]} to {qubit_range_sim[-1]}"
)
circuits_sim[0].draw(output="mpl", fold=-1)

Created 20 circuits with qubit counts from 6 to 25


<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step1-code-1.avif" alt="Output of the previous code cell" />

# Teil I. Qiskit-Patterns
Schauen ma uns jetzt an, wie ma'n KI-Transpiler-Service mit a'm einfachen Quantenschaltkreis verwendet, mithilfe von Qiskit-Patterns. Da Schlüssel is a'n `PassManager` mit `generate_ai_pass_manager()` zu erstellen, anstatt den Standard-`generate_preset_pass_manager()`.
## Schritt 1: Klassische Eingaben auf a Quantenproblem abbilden
In dem Abschnitt testen ma'n KI-Transpiler auf dem `efficient_su2`-Schaltkreis, a'n weit verbreiteten hardware-effizienten Ansatz. Da Schaltkreis is besonders relevant für variationelle Quantenalgorithmen (zum Beispiel VQE) und Quantenmaschinenlern-Aufgaben, was ihn zu a'm idealen Testfall für die Bewertung der Transpilationsleistung macht.

Da `efficient_su2`-Schaltkreis besteht aus abwechselnden Schichten von Einzelqubit-Rotationen und verschränkenden Gates wie CNOTs. Diese Schichten ermöglichen a flexible Erkundung des Quantenzustandsraums, während die Gate-Tiefe überschaubar bleibt. Durch das Optimieren von dem Schaltkreis wollen ma Gate-Anzahl reduzieren, Fidelity verbessern und Rauschen minimieren. Das macht ihn zu a'm starken Kandidaten für's Testen der Effizienz des KI-Transpilers.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    min_num_qubits=100, operational=True, simulator=False
)


pm_default_sim = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    seed_transpiler=seed,
)

In [4]:
results_sim = []

for i, qc in enumerate(circuits_sim):
    n = qubit_range_sim[i]

    qc_default, m_default = transpile_with_metrics(pm_default_sim, qc)

    # Create a fresh AI pass manager each iteration to avoid stale layout state
    pm_ai = generate_ai_pass_manager(
        optimization_level=1,
        ai_optimization_level=3,
        backend=backend,
    )
    qc_ai, m_ai = transpile_with_metrics(pm_ai, qc)

    results_sim.append(
        {
            "Qubits": n,
            "Depth 2Q (Default)": m_default["depth_2q"],
            "Depth 2Q (AI)": m_ai["depth_2q"],
            "Gate Count (Default)": m_default["gate_count"],
            "Gate Count (AI)": m_ai["gate_count"],
            "Time (Default)": m_default["time_s"],
            "Time (AI)": m_ai["time_s"],
        }
    )

print_summary(results_sim)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Metric        Default (mean +/- std)     AI (mean +/- std)      AI % improvement
--------------------------------------------------------------------------------
Depth 2Q               33.0 +/- 12.9          26.4 +/- 8.0      +15.8% +/- 17.6%
Gate Count           522.0 +/- 266.0       560.5 +/- 279.1        -9.0% +/- 9.0%
Time (s)                 0.0 +/- 0.0           0.2 +/- 0.1    -893.6% +/- 362.9%


The summary table shows the mean and standard deviation of each metric across all 20 circuits, along with the average percentage improvement of the AI-powered transpiler over the default. Positive values indicate the AI-powered transpiler produced better results; negative values indicate the default was better.

For this small-scale example, the AI-powered transpiler achieves roughly 16% lower two-qubit depth on average, but at the cost of roughly 9% higher gate count. This highlights a key trade-off when choosing between the two strategies: the AI-powered transpiler prioritizes depth reduction (fewer sequential layers of two-qubit gates), while the default transpiler (SABRE) prioritizes minimizing total gate count (fewer SWAP insertions). Depending on your application, one metric may matter more than the other.

In [5]:
plot_metrics_and_pct(results_sim, "Small-Scale Random Circuits")

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step2-plot-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step2-plot-1.avif" alt="Output of the previous code cell" />

### KI- und traditionelle Pass Manager erstellen
Um die Wirksamkeit des KI-Transpilers zu evaluieren, führen ma zwei Transpilationsdurchläufe durch. Zuerst transpilieren ma'n Schaltkreis mit dem KI-Transpiler. Dann machen ma a'n Vergleich, indem ma denselben Schaltkreis ohne den KI-Transpiler transpilieren, mit traditionellen Methoden. Beide Transpilationsprozesse verwenden dieselbe Coupling Map vom gewählten Backend und den Optimierungsgrad 3 für a'n fairen Vergleich.

Beide Methoden spiegeln den Standardansatz wider, um `PassManager`-Instanzen zu erstellen und Schaltkreise in Qiskit zu transpilieren.

In [6]:
# Use the 10-qubit circuit (index where qubits == 10)
idx_10q = qubit_range_sim.index(10)

qc_10q = circuits_sim[idx_10q]
qc_default_10q, _ = transpile_with_metrics(pm_default_sim, qc_10q)

pm_ai = generate_ai_pass_manager(
    optimization_level=1,
    ai_optimization_level=3,
    backend=backend,
)
qc_ai_10q, _ = transpile_with_metrics(pm_ai, qc_10q)

tqc_methods = {
    "Default": qc_default_10q,
    "AI": qc_ai_10q,
}

print(
    f"Default: depth {qc_default_10q.depth()}, gates {qc_default_10q.size()}"
)
print(f"AI:      depth {qc_ai_10q.depth()}, gates {qc_ai_10q.size()}")

Default: depth 84, gates 280
AI:      depth 91, gates 343


In [7]:
# Build a simple depolarizing noise model
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.001, 1),
    ["sx", "x", "rz"],  # ~0.1% per 1Q gate
)
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.01, 2),
    ["cx", "ecr"],  # ~1% per 2Q gate
)

aer_sim = AerSimulator(noise_model=noise_model)

shots = 10000
survival_probs = {}

for method, tqc in tqc_methods.items():
    mirror = build_mirror_circuit(tqc, simulate=True)

    sampler = SamplerV2(mode=aer_sim)
    job = sampler.run([mirror], shots=shots)
    counts = job.result()[0].data.meas.get_counts()

    all_zeros = "0" * mirror.num_qubits
    survival = counts.get(all_zeros, 0) / shots
    survival_probs[method] = survival
    print(
        f"{method:8s}  P(|00...0>) = {survival:.4f}  "
        f"({counts.get(all_zeros, 0)}/{shots})"
    )

Default   P(|00...0>) = 0.8460  (8460/10000)
AI        P(|00...0>) = 0.8121  (8121/10000)


We ran both mirror circuits through the Aer simulator with a simple depolarizing noise model. The survival probability, defined as the fraction of shots that return the all-zeros bitstring, quantifies how much noise each transpilation strategy introduces.

### Step 4: Post-process and return result in desired classical format

We extract the probability of measuring the all-zeros bitstring from both runs. A higher survival probability indicates better fidelity, meaning the transpilation introduced less noise. The plot below shows the complement, 1 - P(|0...0>), so that a lower bar indicates better fidelity and small differences in error are easier to see.

In [8]:
# Plot 1 - P(|0...0>), the probability of an erroneous (non-zero) outcome.
# A lower bar means the transpilation introduced less noise.
error_probs = {method: 1 - p for method, p in survival_probs.items()}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    error_probs.keys(),
    error_probs.values(),
    color=["steelblue", "coral"],
)
ax.set_ylabel("1 - P(|0...0>)")
ax.set_title("Mirror Circuit Error (10-qubit, Aer Simulator)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step4-code-0.avif" alt="Output of the previous code cell" />

In dem Test vergleichen ma de Leistung vom KI-Transpiler und der Standard-Transpilationsmethode auf dem efficient_su2-Schaltkreis. Da KI-Transpiler erreicht a merklich flachere Schaltkreistiefe bei ähnlicher Gate-Anzahl.

- **Schaltkreistiefe:** Da KI-Transpiler produziert a'n Schaltkreis mit niedrigerer Zwei-Qubit-Tiefe. Das is zu erwarten, da de KI-Passes darauf trainiert san, die Tiefe zu optimieren, indem sie Qubit-Interaktionsmuster lernen und die Hardware-Konnektivität effektiver ausnutzen als regelbasierte Heuristiken.

- **Gate-Anzahl:** De Gesamtzahl der Gates bleibt bei beiden Methoden ähnlich. Das stimmt mit den Erwartungen überein, da de Standard-SABRE-basierte Transpilation explizit de Swap-Anzahl minimiert, die den Gate-Overhead dominiert. Da KI-Transpiler priorisiert stattdessen die Gesamttiefe und kann manchmal a paar zusätzliche Gates für a'n kürzeren Ausführungspfad in Kauf nehmen.

- **Transpilationszeit:** Da KI-Transpiler braucht länger als die Standard-Methode. Das liegt an den zusätzlichen Rechenkosten durch das Aufrufen von gelernten Modellen während Routing und Synthese. Im Vergleich dazu is da SABRE-basierte Transpiler nach seiner Neuschreibung und Optimierung in Rust deutlich schneller geworden und bietet hocheffizientes heuristisches Routing in großem Maßstab.

Wichtig is zu beachten, dass diese Ergebnisse auf nur a'm Schaltkreis basieren. Um a'n umfassendes Verständnis davon zu bekommen, wie der KI-Transpiler mit traditionellen Methoden verglichen werden kann, is es notwendig, a Vielzahl von Schaltkreisen zu testen. Die Leistung von QTS kann je nach Art des zu optimierenden Schaltkreises stark variieren. Für a'n umfassenderen Vergleich schau dir de Benchmarks weiter oben an oder besuch den [Blog-Beitrag.](https://www.ibm.com/quantum/blog/qiskit-performance)
## Schritt 3: Mit Qiskit-Primitiven ausführen
Da sich dieses Tutorial auf Transpilation konzentriert, werden keine Experimente auf dem Quantengerät ausgeführt. Das Ziel is, de Optimierungen aus Schritt 2 zu nutzen, um a'n transpilierten Schaltkreis mit reduzierter Tiefe oder Gate-Anzahl zu erhalten.
## Schritt 4: Nachbearbeiten und Ergebnis im gewünschten klassischen Format zurückgeben
Da für dieses Notebook keine Ausführung stattfindet, gibt's keine Ergebnisse zum Nachbearbeiten.
# Teil II. Transpilierte Schaltkreise analysieren und benchmarken
In dem Abschnitt zeigen ma, wie ma'n transpilierten Schaltkreis analysiert und ihn detaillierter gegen die Originalversion benchmarkt. Ma konzentrieren uns auf Metriken wie Schaltkreistiefe, Gate-Anzahl und Transpilationszeit, um die Effektivität der Optimierung zu beurteilen. Außerdem besprechen ma, wie de Ergebnisse bei verschiedenen Schaltkreistypen unterschiedlich sein können, was Einblicke in die breitere Leistung des Transpilers in verschiedenen Szenarien bietet.

In [9]:
# -------------------------Step 1-------------------------
num_circuits_hw = 25
depth_hw = 8
qubit_range_hw = list(range(26, 51))

circuits_hw = [
    # We have only two qubit gates, as those test how well the
    # transpiler can optimize the circuit.
    random_circuit(
        num_qubits=n,
        depth=depth_hw,
        max_operands=2,
        num_operand_distribution={2: 1},
        seed=seed + i,
    )
    for i, n in enumerate(qubit_range_hw)
]

print(
    f"Created {len(circuits_hw)} circuits with qubit counts "
    f"from {qubit_range_hw[0]} to {qubit_range_hw[-1]}"
)

Created 25 circuits with qubit counts from 26 to 50


In [10]:
# -------------------------Step 2-------------------------
pm_default = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    seed_transpiler=seed,
)

results_hw = []

for i, qc in enumerate(circuits_hw):
    n = qubit_range_hw[i]

    qc_default, m_default = transpile_with_metrics(pm_default, qc)

    # Create a fresh AI pass manager each iteration to avoid stale layout state
    pm_ai = generate_ai_pass_manager(
        optimization_level=1,
        ai_optimization_level=3,
        backend=backend,
    )
    qc_ai, m_ai = transpile_with_metrics(pm_ai, qc)

    results_hw.append(
        {
            "Qubits": n,
            "Depth 2Q (Default)": m_default["depth_2q"],
            "Depth 2Q (AI)": m_ai["depth_2q"],
            "Gate Count (Default)": m_default["gate_count"],
            "Gate Count (AI)": m_ai["gate_count"],
            "Time (Default)": m_default["time_s"],
            "Time (AI)": m_ai["time_s"],
        }
    )

print_summary(results_hw)

Metric        Default (mean +/- std)     AI (mean +/- std)      AI % improvement
--------------------------------------------------------------------------------
Depth 2Q              217.4 +/- 50.4        191.0 +/- 35.6      +10.9% +/- 10.7%
Gate Count         4513.3 +/- 1394.3     5227.1 +/- 1536.4       -16.4% +/- 5.8%
Time (s)                 0.1 +/- 0.0           3.5 +/- 1.5   -3588.2% +/- 643.6%


In [11]:
plot_metrics_and_pct(results_hw, "Large-Scale Random Circuits")

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step2-plot-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step2-plot-1.avif" alt="Output of the previous code cell" />

In [12]:
# -------------------------Step 3-------------------------
# Build mirror circuits from the 26-qubit case
idx_26q = qubit_range_hw.index(26)

qc_26q = circuits_hw[idx_26q]
qc_default_26q, _ = transpile_with_metrics(pm_default, qc_26q)

pm_ai = generate_ai_pass_manager(
    optimization_level=1,
    ai_optimization_level=3,
    backend=backend,
)
qc_ai_26q, _ = transpile_with_metrics(pm_ai, qc_26q)

mirror_default_hw = build_mirror_circuit(qc_default_26q, simulate=False)
mirror_ai_hw = build_mirror_circuit(qc_ai_26q, simulate=False)

# Re-transpile to basis gates (the inverse can introduce gates like sxdg)
pm_basis = generate_preset_pass_manager(
    optimization_level=0,
    backend=backend,
)
mirror_default_hw = pm_basis.run(mirror_default_hw)
mirror_ai_hw = pm_basis.run(mirror_ai_hw)

print(
    f"Mirror circuit (Default): depth {mirror_default_hw.depth()}, "
    f"gates {mirror_default_hw.size()}"
)
print(
    f"Mirror circuit (AI):      depth {mirror_ai_hw.depth()}, "
    f"gates {mirror_ai_hw.size()}"
)

# Submit to real hardware
sampler_hw = SamplerV2(mode=backend)
sampler_hw.options.environment.job_tags = ["TUT_AITI"]

shots_hw = 500000
job_hw = sampler_hw.run([mirror_default_hw, mirror_ai_hw], shots=shots_hw)
print(f"Job submitted: {job_hw.job_id()}")

Mirror circuit (Default): depth 1577, gates 9672
Mirror circuit (AI):      depth 1235, gates 11092
Job submitted: d8gt7vm6983c73dqbg0g


In [13]:
# -------------------------Step 4-------------------------
result_hw = job_hw.result()

survival_probs_hw = {}
for i, method in enumerate(["Default", "AI"]):
    counts = result_hw[i].data.meas.get_counts()
    mirror = [mirror_default_hw, mirror_ai_hw][i]
    all_zeros = "0" * mirror.num_qubits
    survival = counts.get(all_zeros, 0) / shots_hw
    survival_probs_hw[method] = survival
    print(
        f"{method:8s}  P(|00...0>) = {survival:.4f}  "
        f"({counts.get(all_zeros, 0)}/{shots_hw})"
    )

# Plot 1 - P(|0...0>), the probability of an erroneous (non-zero) outcome.
# A lower bar means the transpilation introduced less noise.
error_probs_hw = {method: 1 - p for method, p in survival_probs_hw.items()}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    error_probs_hw.keys(),
    error_probs_hw.values(),
    color=["steelblue", "coral"],
)
ax.set_ylabel("1 - P(|0...0>)")
ax.set_title(f"Mirror Circuit Error (26-qubit, {backend.name})")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

Default   P(|00...0>) = 0.0005  (239/500000)
AI        P(|00...0>) = 0.0050  (2516/500000)


<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step4-1.avif" alt="Output of the previous code cell" />

### Analysis of results

The large-scale results reinforce the trends observed in the small-scale example, now at a more demanding scale.

**Two-qubit depth:** The AI-powered transpiler continues to deliver noticeably lower two-qubit depth across the full range of circuit sizes. Depth optimization is one of the primary objectives the AI routing model is trained on, and the advantage is more pronounced at larger qubit counts where the routing problem becomes harder for heuristic methods.

**Gate count:** The default transpiler (SABRE) consistently produces circuits with fewer gates across all circuit sizes in this range. SABRE's heuristic is specifically designed to minimize gate count, and at this scale the advantage is clear and uniform.

**Transpilation time:** The gap in transpilation time widens at larger scales. SABRE remains nearly constant, while the AI-powered transpiler's runtime grows more steeply. Despite this, the AI-powered transpiler runtime remains practical for most workflows.

**Mirror circuit fidelity:** Both methods produce survival probabilities well under 1% at this scale, leaving little usable signal. With total gate counts around 10,000 and two-qubit depths exceeding 1,000, the depolarizing noise accumulated across the mirror circuit overwhelms most of the signal. This highlights a key limitation of the mirror circuit approach: while it is simple and requires no classical simulation, it does not scale well to large or deep circuits, where both methods are pushed close to the noise floor and the small surviving signal is dominated by accumulated error.

While these results underscore the AI-powered transpiler's effectiveness, it is important to note its limitations. The AI synthesis method is currently only available for certain coupling maps, which may restrict its broader applicability. This constraint should be considered when evaluating its usage in different scenarios.

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Transpilation optimizations with SABRE](/docs/tutorials/transpilation-optimizations-with-sabre)
- [Compilation methods for Hamiltonian simulation circuits](/docs/tutorials/compilation-methods-for-hamiltonian-simulation-circuits)

</Admonition>